# 官方成交量拆分 — 最小可行試抓與對帳（Pilot）

- **第 5 格（試抓）**：向證交所和櫃買中心的公開報表抓 8 個試點日，每個請求都保存原始回應和 SHA-256。
  - 抓取報表：每日收盤行情、盤後定價、零股、鉅額。
  - 端點都是「候選」，抓不到也算結果。
  - 不需要任何憑證，不呼叫富邦。
- **第 6 格（對帳）**：只讀既有的缺棒帳本，不讀寫任何資料集。
  - 先在「K 棒全到齊」的日子實測：官方總量扣掉哪些成分，才會剛好等於 5 分K加總。
  - 這個關係驗證通過後，才把有缺棒的日子分類。分類結果都只是候選，帳本狀態不變。
- 結果寫到 Colab 本機，再複製到 Drive 的 `_tw-market-data-lake/audits/official_volume_pilot_<時間>/`（新資料夾）。

In [ ]:
# 1. 參數
DATES = "2023-08-17,2024-01-02,2024-05-24,2024-12-16,2025-04-10,2025-06-23,2026-03-30,2026-06-23"
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
LEDGER_DIR = f"{DATA_ROOT}/_tw-market-data-lake/audits/gap_ledger_20261026_163358_3e8eed0"
OUT_DIR = "/content/tw_lake_official_volume"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# 4. 依賴
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)


In [ ]:
# 5. 官方報表試抓（約 8 日 x 11 個候選端點，含禮貌間隔約 5 分鐘）
import glob
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--dates", DATES, "--out-dir", OUT_DIR]))
PROBE_DIR = sorted(glob.glob(f"{OUT_DIR}/official_volume_probe_*"))[-1]

In [ ]:
# 6. 對帳（唯讀）
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_DIR,
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]

In [ ]:
# 7. 複製到 Drive（新資料夾）並等待上傳
ensure_drive()
import shutil, os
dest = f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_{os.path.basename(PROBE_DIR).split('probe_')[1]}"
shutil.copytree(PROBE_DIR, f"{dest}/{os.path.basename(PROBE_DIR)}")
shutil.copytree(RECON_DIR, f"{dest}/{os.path.basename(RECON_DIR)}")
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()   # last step only; re-run ensure_drive() before using Drive again
print("Drive flushed")

## 8–10. 第二輪：單位修正後重新解析＋重新對帳＋櫃買端點探索（不用重抓證交所）
- **第 8 格**：用已保存的原始回應重新解析，不再發出網路請求。
  - 單位依「成交金額 ÷（數量 × 價格）」實際推算，推不出來才看報表註記。例如盤後定價「成交數量」的單位是千股。
  - 結果寫到原 probe 資料夾下新的 `reparse_*`，原始檔不動。
- **第 9 格**：用新的解析結果重新對帳。
- **第 10 格**：下載櫃買中心的 OpenAPI 規格和網站腳本，列出所有 API 路徑，用來找盤後定價、零股、鉅額的正確端點。

In [ ]:
# 8. 重新解析（無網路）。新開的 runtime 請先跑第 1、3、4 格
import glob, os, shutil, sys
ensure_drive()
PROBE_ON_DRIVE = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_*/official_volume_probe_*"))[-1]
print("probe:", PROBE_ON_DRIVE)
local_probe = f"{OUT_DIR}/{os.path.basename(PROBE_ON_DRIVE)}"
if not os.path.exists(local_probe):
    shutil.copytree(PROBE_ON_DRIVE, local_probe)
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--reparse", local_probe]))
REPARSE_DIR = sorted(glob.glob(f"{local_probe}/reparse_*"))[-1]

In [ ]:
# 9. 重新對帳（唯讀）
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", local_probe,
                      "--parsed-file", f"{REPARSE_DIR}/parsed_volume.parquet",
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON2_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]

In [ ]:
# 10. 櫃買中心端點探索（3 個請求）＋ 複製第二輪結果到 Drive
ensure_drive()
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--discover-tpex", "--out-dir", OUT_DIR]))
DISC_DIR = sorted(glob.glob(f"{OUT_DIR}/tpex_discovery_*"))[-1]
dest = os.path.dirname(PROBE_ON_DRIVE)
for d in (REPARSE_DIR, RECON2_DIR, DISC_DIR):
    shutil.copytree(d, f"{dest}/round2/{os.path.basename(d)}")
print("copied to", f"{dest}/round2")
from google.colab import drive
drive.flush_and_unmount()   # flush uploads; the next cell re-mounts with ensure_drive()


## 11–13. 第三輪：逐日驗證＋櫃買中心頁面探索（先跑第 1、3、4 格）
- **第 11 格**：用第二輪的解析結果重新對帳，不發網路請求。
  - 新增每個交易所、每一天的驗證表 `identity_by_date`。
  - 只有當天的完整日能對上（≥99%）才分類；對不上的日期列為異常待查，不分類。
- **第 12 格**：讀櫃買中心網站地圖，打開「盤後定價／零股／鉅額／收盤行情」相關頁面，列出各頁面使用的資料端點。約 20–40 個請求，每頁間隔 1.5 秒。
- **第 13 格**：把結果複製到 Drive 的 `round3/`。

In [ ]:
# 11. 逐日驗證對帳（無網路）
import glob, os, shutil, sys
ensure_drive()
PILOT = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_*"))[-1]
PROBE_ON_DRIVE = sorted(glob.glob(f"{PILOT}/official_volume_probe_*"))[-1]
PARSED = sorted(glob.glob(f"{PILOT}/round2/reparse_*/parsed_volume.parquet"))[-1]
print("probe:", PROBE_ON_DRIVE, "\nparsed:", PARSED)
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_ON_DRIVE,
                      "--parsed-file", PARSED, "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON3_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]
import pandas as pd
print(pd.read_csv(f"{RECON3_DIR}/identity_by_date.csv").to_string(index=False))

In [ ]:
# 12. 櫃買中心頁面探索
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--discover-tpex", "--out-dir", OUT_DIR]))
DISC3_DIR = sorted(glob.glob(f"{OUT_DIR}/tpex_discovery_*"))[-1]

In [ ]:
# 13. 複製到 Drive
ensure_drive()
for d in (RECON3_DIR, DISC3_DIR):
    shutil.copytree(d, f"{PILOT}/round3/{os.path.basename(d)}")
print("copied to", f"{PILOT}/round3")
from google.colab import drive
drive.flush_and_unmount()   # flush uploads; the next cell re-mounts with ensure_drive()


## 14–16. 第四輪：櫃買中心成分試抓＋合併對帳（先跑第 1、3、4 格）
- **第 14 格**：只抓上一輪找到的 6 個櫃買中心端點，日期同前 8 天，共 48 個請求，每個間隔 2.5 秒，約 2 分鐘。
  - 端點：收盤行情（不含定價）、盤後定價、盤中零股、盤後零股、個股鉅額、鉅額日成交。
- **第 15 格**：把這次結果和第二輪的證交所解析合併，重新逐日對帳。不發網路請求。
- **第 16 格**：把結果複製到 Drive 的 `round4/`。

In [ ]:
# 14. 櫃買中心成分試抓
import glob, os, shutil, sys
ensure_drive()
ONLY = "otc_EW,fixPricing,oddQuote,odd,blockTrade_tradeInfo,blockTrade_quote"
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--dates", DATES, "--only", ONLY,
                      "--out-dir", OUT_DIR]))
PROBE4_DIR = sorted(glob.glob(f"{OUT_DIR}/official_volume_probe_*"))[-1]
print(PROBE4_DIR)

In [ ]:
# 15. 合併對帳（無網路）
PILOT = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_*"))[-1]
PROBE_ON_DRIVE = sorted(glob.glob(f"{PILOT}/official_volume_probe_*"))[-1]
PARSED = sorted(glob.glob(f"{PILOT}/round2/reparse_*/parsed_volume.parquet"))[-1]
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_ON_DRIVE,
                      "--parsed-file", PARSED, "--extra-probe-dirs", PROBE4_DIR,
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON4_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]
import pandas as pd
print(pd.read_csv(f"{RECON4_DIR}/identity_by_date.csv").to_string(index=False))
print(pd.read_csv(f"{RECON4_DIR}/identity_tests.csv").query("exchange == 'TPEX'").sort_values("exact_pct", ascending=False).head(8).to_string(index=False))

In [ ]:
# 16. 複製到 Drive
ensure_drive()
for d in (PROBE4_DIR, RECON4_DIR):
    shutil.copytree(d, f"{PILOT}/round4/{os.path.basename(d)}")
print("copied to", f"{PILOT}/round4")
from google.colab import drive
drive.flush_and_unmount()

## 17–21. 第五輪：櫃買中心解析修正＋盤後零股參數＋富邦交叉比對（先跑第 1、3、4 格）
- **第 17 格**：用已保存的第四輪回應重新解析（盤後定價讀「成交張數」，個股鉅額只取當天），不發網路請求。接著試盤後零股的兩組參數，共 16 個請求。
- **第 18 格**：把證交所和櫃買中心的資料合併，逐日重新對帳。不發網路請求。
- **第 19 格**：設定富邦憑證（Colab Secrets）並安裝 SDK，和 Runner 相同。
- **第 20 格**：富邦交叉比對（只讀行情）。
  - 抽樣：證交所每個未驗證日期抽 20 個「K 棒全到齊但量對不上」的股票日，取殘差最大的 10 個和隨機 10 個；每個已驗證日期另抽 5 個對照。
  - 每個股票日向富邦重新抓日K、5 分K、1 分K。共約 115 個股票日 × 3 ≈ 350 個請求，約 7 分鐘。
  - 用途：判斷少掉的量是我們存的資料有誤、富邦 5 分K 漏量，還是富邦整體沒有這些成交。
- **第 21 格**：把第五輪所有結果複製到 Drive 的 `round5/`，最後卸載 Drive。

In [ ]:
# 17. 重新解析第四輪＋盤後零股參數試抓
import glob, os, shutil, sys
ensure_drive()
PILOT = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_*"))[-1]
P4 = sorted(glob.glob(f"{PILOT}/round4/official_volume_probe_*"))[-1]
local_p4 = f"{OUT_DIR}/r4_{os.path.basename(P4)}"
if not os.path.exists(local_p4):
    shutil.copytree(P4, local_p4)
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--reparse", local_p4]))
REPARSE4 = sorted(glob.glob(f"{local_p4}/reparse_*"))[-1]
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--dates", DATES,
                      "--only", "odd_noType,odd_AL", "--out-dir", OUT_DIR]))
PROBE5_DIR = sorted(glob.glob(f"{OUT_DIR}/official_volume_probe_*"))[-1]

In [ ]:
# 18. 合併對帳（無網路）
PROBE_ON_DRIVE = sorted(glob.glob(f"{PILOT}/official_volume_probe_*"))[-1]
PARSED = sorted(glob.glob(f"{PILOT}/round2/reparse_*/parsed_volume.parquet"))[-1]
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_ON_DRIVE,
                      "--parsed-file", PARSED, "--extra-probe-dirs", f"{REPARSE4},{PROBE5_DIR}",
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON5_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]
import pandas as pd
print(pd.read_csv(f"{RECON5_DIR}/identity_by_date.csv").to_string(index=False))
print(pd.read_csv(f"{RECON5_DIR}/identity_tests.csv").query("exchange == 'TPEX'").sort_values("exact_pct", ascending=False).head(8).to_string(index=False))

In [ ]:
# 19. 富邦憑證（Colab Secrets，值不會被印出）＋ SDK
from google.colab import userdata
FUBON_PFX_DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/Fubon_Cert/your_cert.pfx"   # 改成實際位置
for k, req in (("FUBON_USER_ID", True), ("FUBON_PASSWORD", False), ("FUBON_CERT_PASSWORD", False), ("FUBON_API_KEY", False)):
    try:
        v = userdata.get(k)
    except Exception:
        v = None
    if req and not v:
        raise RuntimeError(f"缺少 Colab Secret: {k}")
    os.environ[k] = v or ""
ensure_drive()
assert os.path.exists(FUBON_PFX_DRIVE_PATH), "找不到憑證檔，請修改 FUBON_PFX_DRIVE_PATH"
os.environ["FUBON_CERT_PATH"] = FUBON_PFX_DRIVE_PATH
subprocess.run([sys.executable, f"{PROJECT_DIR}/scripts/install_colab_dependencies.py"], check=True)

In [ ]:
# 20. 富邦交叉比對（只讀行情端點）
ensure_drive()
print("exit", stream([sys.executable, "-m", "tw_lake.audit.provider_crosscheck", "--data-root", DATA_ROOT,
                      "--recon-dir", RECON5_DIR, "--out-dir", OUT_DIR, "--per-date", "20", "--control-per-date", "5"]))
CROSS_DIR = sorted(glob.glob(f"{OUT_DIR}/provider_crosscheck_*"))[-1]

In [ ]:
# 21. 複製第五輪結果到 Drive
ensure_drive()
for d in (REPARSE4, PROBE5_DIR, RECON5_DIR, CROSS_DIR):
    shutil.copytree(d, f"{PILOT}/round5/{os.path.basename(d)}")
print("copied to", f"{PILOT}/round5")
from google.colab import drive
drive.flush_and_unmount()

## 22–25. 第六輪：櫃買盤後零股參數＋補抓盤後定價缺的 2 天＋證交所盤中分段量（先跑第 1、3、4 格）
- **第 22 格**：下載櫃買「盤後零股」頁面和它的腳本（約 3–6 個請求），印出網頁實際呼叫的端點與參數，不再猜參數。
- **第 23 格**：試抓公開報表，8 天 × 5 個端點，共 40 個請求，約 2–3 分鐘。
  - 櫃買盤後定價：重抓，補第四輪缺的 2 天。
  - 櫃買盤後零股：三組參數。
  - 證交所 MI_5MINS：全市場每 5 秒累積成交量。
- **第 24 格**：合併第二、四、五、六輪重新對帳，接著做盤中分段量比對（只讀 5 分 K）。
  - 每 5 分鐘一格，比較官方累積量增量與富邦 5 分 K 加總，找出未驗證日缺量落在哪個時段，例如 13:30 收盤集合競價之後。
  - 不發網路請求，不改任何分類。
- **第 25 格**：把結果複製到 Drive 的 `round6/`。

In [ ]:
# 22. 櫃買盤後零股頁面檢查
import glob, os, shutil, sys
import pandas as pd
ensure_drive()
PILOT = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_*"))[-1]
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--inspect-tpex-page",
                      "https://www.tpex.org.tw/zh-tw/mainboard/trading/info/odd-lot/post-pricing/day.html",
                      "--out-dir", OUT_DIR]))
INSPECT_DIR = sorted(glob.glob(f"{OUT_DIR}/tpex_page_*"))[-1]

In [ ]:
# 23. 試抓：盤後定價補抓＋盤後零股三組參數＋MI_5MINS
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--dates", DATES,
                      "--only", "fixPricing,odd_roc,odd_d8,odd_roc_EW,MI_5MINS", "--out-dir", OUT_DIR]))
PROBE6_DIR = sorted(glob.glob(f"{OUT_DIR}/official_volume_probe_*"))[-1]
rq = pd.read_parquet(f"{PROBE6_DIR}/requests.parquet")
print(rq.pivot_table(index="date", columns="report", values="parsed_rows", aggfunc="max").to_string())
print(rq[["report", "date", "http_status", "stat", "note"]].query("report != 'MI_5MINS'").head(40).to_string(index=False))

In [ ]:
# 24. 合併對帳＋盤中分段量（無網路）
PROBE_ON_DRIVE = sorted(glob.glob(f"{PILOT}/official_volume_probe_*"))[-1]
PARSED = sorted(glob.glob(f"{PILOT}/round2/reparse_*/parsed_volume.parquet"))[-1]
R5 = f"{PILOT}/round5"
REPARSE4_D = sorted(glob.glob(f"{R5}/reparse_*"))[-1]
PROBE5_D = sorted(glob.glob(f"{R5}/official_volume_probe_*"))[-1]
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_ON_DRIVE,
                      "--parsed-file", PARSED, "--extra-probe-dirs", f"{REPARSE4_D},{PROBE5_D},{PROBE6_DIR}",
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON6_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]
print(pd.read_csv(f"{RECON6_DIR}/identity_by_date.csv").to_string(index=False))
print(pd.read_csv(f"{RECON6_DIR}/identity_tests.csv").query("exchange == 'TPEX'").sort_values("exact_pct", ascending=False).head(8).to_string(index=False))
print("exit", stream([sys.executable, "-m", "tw_lake.audit.intraday_profile", "--data-root", DATA_ROOT,
                      "--probe-dir", PROBE6_DIR, "--recon-dir", RECON6_DIR, "--out-dir", OUT_DIR]))
PROFILE_DIR = sorted(glob.glob(f"{OUT_DIR}/intraday_profile_*"))[-1]

In [ ]:
# 25. 複製第六輪結果到 Drive
ensure_drive()
for d in (INSPECT_DIR, PROBE6_DIR, RECON6_DIR, PROFILE_DIR):
    shutil.copytree(d, f"{PILOT}/round6/{os.path.basename(d)}")
print("copied to", f"{PILOT}/round6")
from google.colab import drive
drive.flush_and_unmount()

## 26–28. 第七輪：櫃買盤後零股（type=Daily）＋全部合併對帳＋新分類（先跑第 1、3、4 格，約 5 分鐘）
- **第 26 格**：只抓櫃買盤後零股，8 個請求，約 25 秒。參數取自網頁表單：`type=Daily`，日期格式 `YYYY/MM/DD`。
- **第 27 格**：把第二、四、五、六、七輪合併重新對帳，不發網路請求。
  - 新增「只有整張短少」日期型態：當天對不上的完整日全都是官方比 5 分 K 多出整張，沒有反向。
  - 在這種日子，個股自己能對上 → `CANDIDATE_NO_TRADE_SYMBOL_EXACT_DATE_WHOLE_LOT_SHORT`（證據強度低於驗證日）。
  - 在這種日子，個股短少整張 → `DAY_LEVEL_MISSING_VOLUME_ON_WHOLE_LOT_SHORT_DATE`（日層級，定不到格）。
  - 所有分類都只是候選，帳本狀態不變。
- **第 28 格**：複製結果到 Drive 的 `round7/`。

In [ ]:
# 26. 櫃買盤後零股（type=Daily）
import glob, os, shutil, sys, json
import pandas as pd
ensure_drive()
PILOT = sorted(glob.glob(f"{DATA_ROOT}/_tw-market-data-lake/audits/official_volume_pilot_*"))[-1]
print("exit", stream([sys.executable, "-m", "tw_lake.reference.official_volume", "--dates", DATES,
                      "--only", "odd_Daily", "--out-dir", OUT_DIR]))
PROBE7_DIR = sorted(glob.glob(f"{OUT_DIR}/official_volume_probe_*"))[-1]
print(pd.read_parquet(f"{PROBE7_DIR}/requests.parquet")[["date", "http_status", "stat", "parsed_rows", "share_fields"]].to_string(index=False))

In [ ]:
# 27. 全部合併對帳（無網路）
PROBE_ON_DRIVE = sorted(glob.glob(f"{PILOT}/official_volume_probe_*"))[-1]
PARSED = sorted(glob.glob(f"{PILOT}/round2/reparse_*/parsed_volume.parquet"))[-1]
EXTRA = [sorted(glob.glob(f"{PILOT}/round5/reparse_*"))[-1],
         sorted(glob.glob(f"{PILOT}/round5/official_volume_probe_*"))[-1],
         sorted(glob.glob(f"{PILOT}/round6/official_volume_probe_*"))[-1], PROBE7_DIR]
print("exit", stream([sys.executable, "-m", "tw_lake.audit.volume_recon", "--probe-dir", PROBE_ON_DRIVE,
                      "--parsed-file", PARSED, "--extra-probe-dirs", ",".join(EXTRA),
                      "--ledger-dir", LEDGER_DIR, "--out-dir", OUT_DIR]))
RECON7_DIR = sorted(glob.glob(f"{OUT_DIR}/volume_recon_*"))[-1]
pd.set_option("display.width", 250)
print(pd.read_csv(f"{RECON7_DIR}/identity_by_date.csv").to_string(index=False))
print(pd.read_csv(f"{RECON7_DIR}/identity_tests.csv").query("exchange == 'TPEX'").sort_values("exact_pct", ascending=False).head(6).to_string(index=False))
s = json.load(open(f"{RECON7_DIR}/volume_recon_summary.json"))
print(pd.DataFrame(s["missing_days"]["by_class"]).to_string(index=False))
print({k: v for k, v in s["missing_days"].items() if k != "by_class"})
print(pd.DataFrame(s["complete_days_whole_lot_short"]).to_string(index=False))

In [ ]:
# 28. 複製第七輪結果到 Drive
ensure_drive()
for d in (PROBE7_DIR, RECON7_DIR):
    shutil.copytree(d, f"{PILOT}/round7/{os.path.basename(d)}")
print("copied to", f"{PILOT}/round7")
from google.colab import drive
drive.flush_and_unmount()